In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import IterableDataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from datasets import load_dataset
from PIL import Image
import numpy as np
import wandb
from tqdm import tqdm

# For metrics & viz:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

Load Dataset

In [2]:
import datasets
datasets.config.HF_DATASETS_OFFLINE = False
datasets.config.STREAMING_READ_MAX_RETRIES = 10

ds = load_dataset(
    "HichTala/coco-background",
    streaming=True
)

ds = ds.shuffle(buffer_size=50_000, seed=42)

print(ds)
print(ds["train"].features)
print(ds["validation"].features)
print("num train (for streaming, this might not be precise):")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Resolving data files:   0%|          | 0/103 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/103 [00:00<?, ?it/s]

IterableDatasetDict({
    train: IterableDataset({
        features: ['image', 'label'],
        num_shards: 103
    })
    validation: IterableDataset({
        features: ['image', 'label'],
        num_shards: 5
    })
})
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['airplane', 'apple', 'background', 'backpack', 'banana', 'baseball bat', 'baseball glove', 'bear', 'bed', 'bench', 'bicycle', 'bird', 'boat', 'book', 'bottle', 'bowl', 'broccoli', 'bus', 'cake', 'car', 'carrot', 'cat', 'cell phone', 'chair', 'clock', 'couch', 'cow', 'cup', 'dining table', 'dog', 'donut', 'elephant', 'fire hydrant', 'fork', 'frisbee', 'giraffe', 'hair drier', 'handbag', 'horse', 'hot dog', 'keyboard', 'kite', 'knife', 'laptop', 'microwave', 'motorcycle', 'mouse', 'orange', 'oven', 'parking meter', 'person', 'pizza', 'potted plant', 'refrigerator', 'remote', 'sandwich', 'scissors', 'sheep', 'sink', 'skateboard', 'skis', 'snowboard', 'spoon', 'sports ball', 'stop sign', 'suitcase', 'su

In [3]:
'''from datasets import load_dataset
import random
from collections import Counter

# Load dataset in streaming mode
ds = load_dataset(
    "HichTala/coco-background",
    split="train",
    streaming=True
)

# Shuffle with a large buffer to avoid reading only the beginning
ds = ds.shuffle(buffer_size=20_000, seed=42)

labels = []
max_samples = 1_000   # how many samples we probe
stop_if_diverse = True

for i, ex in enumerate(ds):
    labels.append(ex["label"])

    # Early stop if we already see diversity
    if stop_if_diverse and len(set(labels)) > 1:
        break

    if i + 1 >= max_samples:
        break

counter = Counter(labels)

print("Number of samples checked:", len(labels))
print("Number of unique labels:", len(counter))
print("Label distribution (first 10):", counter.most_common(10))'''

'from datasets import load_dataset\nimport random\nfrom collections import Counter\n\n# Load dataset in streaming mode\nds = load_dataset(\n    "HichTala/coco-background",\n    split="train",\n    streaming=True\n)\n\n# Shuffle with a large buffer to avoid reading only the beginning\nds = ds.shuffle(buffer_size=20_000, seed=42)\n\nlabels = []\nmax_samples = 1_000   # how many samples we probe\nstop_if_diverse = True\n\nfor i, ex in enumerate(ds):\n    labels.append(ex["label"])\n\n    # Early stop if we already see diversity\n    if stop_if_diverse and len(set(labels)) > 1:\n        break\n\n    if i + 1 >= max_samples:\n        break\n\ncounter = Counter(labels)\n\nprint("Number of samples checked:", len(labels))\nprint("Number of unique labels:", len(counter))\nprint("Label distribution (first 10):", counter.most_common(10))'

In [4]:
'''it = iter(ds["train"])
labels = []
for _ in range(200):
    labels.append(next(it)["label"])
print("unique labels in first 200:", sorted(set(labels))[:50])
print("num unique:", len(set(labels)))'''

'it = iter(ds["train"])\nlabels = []\nfor _ in range(200):\n    labels.append(next(it)["label"])\nprint("unique labels in first 200:", sorted(set(labels))[:50])\nprint("num unique:", len(set(labels)))'

In [5]:
'''ex = next(iter(ds["train"]))
print(ex.keys())
print("label raw:", ex["label"], type(ex["label"]))'''

'ex = next(iter(ds["train"]))\nprint(ex.keys())\nprint("label raw:", ex["label"], type(ex["label"]))'

In [6]:
IMAGE_KEY = "image"
LABEL_KEY = "label"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Transforms
train_tfms = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])



In [7]:
class HFImageIterableDataset(IterableDataset):
    def __init__(self, hf_split, transform=None, max_samples=None):
        """
        Args:
            hf_split: HuggingFace streaming dataset split
            transform: torchvision transforms
            max_samples: Maximum samples per epoch
        """
        self.hf_split = hf_split
        self.transform = transform
        self.max_samples = max_samples

    def __iter__(self):
        count = 0
        try:
            for ex in self.hf_split:
                # Check max samples limit
                if self.max_samples and count >= self.max_samples:
                    break

                try:
                    # Process image
                    img = ex[IMAGE_KEY]
                    if not isinstance(img, Image.Image):
                        img = Image.fromarray(np.array(img))
                    img = img.convert("RGB")

                    # Get label
                    y = ex[LABEL_KEY]

                    # Apply transforms
                    if self.transform:
                        img = self.transform(img)

                    count += 1
                    yield img, y

                except Exception as e:
                    print(f"Error processing sample {count}: {e}")
                    continue

        except Exception as e:
            print(f"Error in dataset iteration: {e}")
            raise

  # def get_num_classes(ds_split):
  #   """Extract number of classes from dataset features"""
  #   if LABEL_KEY in ds_split.features:
  #       feature = ds_split.features[LABEL_KEY]
  #       if hasattr(feature, 'num_classes'):
  #           return feature.num_classes
  #   return None

In [8]:
def build_loaders(ds, cfg):
    train_split = ds["train"].shuffle(seed=cfg["seed"], buffer_size=cfg["shuffle_buffer"])
    val_split = ds["validation"]

    train_samples = cfg["steps_per_epoch"] * cfg["batch_size"]
    val_samples = cfg["val_steps"] * cfg["batch_size"]

    train_ds = HFImageIterableDataset(train_split, transform=train_tfms, max_samples=train_samples)
    val_ds = HFImageIterableDataset(val_split, transform=val_tfms, max_samples=val_samples)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg["batch_size"],
        num_workers=0,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg["batch_size"],
        num_workers=0,
        pin_memory=True
    )
    return train_loader, val_loader

Config Wandb

In [9]:
config = {
    "batch_size": 8,
    "lr": 1e-4,
    "epochs": 2,
    "steps_per_epoch": 10,  
    "val_steps": 5,
    "shuffle_buffer": 200,
    "dataset": "HichTala/coco-background",
    "seed": 42
}

'''wandb.init(
    project="domain-shift-coco-dota",
    name="resnet50_finetuned_on_coco",
    config=config
)'''

'''# Optional: make W&B x-axis consistent and avoid “step vs step” plots
wandb.define_metric("global_step")
wandb.define_metric("train/*", step_metric="global_step")
wandb.define_metric("val/*", step_metric="global_step")
wandb.define_metric("epoch/*", step_metric="global_step")'''


'# Optional: make W&B x-axis consistent and avoid “step vs step” plots\nwandb.define_metric("global_step")\nwandb.define_metric("train/*", step_metric="global_step")\nwandb.define_metric("val/*", step_metric="global_step")\nwandb.define_metric("epoch/*", step_metric="global_step")'

In [10]:
num_classes = ds["train"].features["label"].num_classes
class_names = ds["train"].features["label"].names

print("num_classes:", num_classes)
print("first 5 classes:", class_names[:5])

train_loader, val_loader = build_loaders(ds, config)

# --- Load pretrained backbone
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

# --- Replace classifier
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)

# --- FREEZE EVERYTHING
for param in model.parameters():
    param.requires_grad = False

# --- UNFREEZE layer4 + classifier ONLY
for param in model.layer4.parameters():
    param.requires_grad = True

for param in model.fc.parameters():
    param.requires_grad = True

# --- Device
model = model.to(DEVICE)
print("DEVICE:", DEVICE)
print("Model device:", next(model.parameters()).device)

# --- Sanity check batch
xb, yb = next(iter(train_loader))
xb = xb.to(DEVICE)
yb = yb.to(DEVICE)
print("Batch device:", xb.device)

# --- Loss
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# --- Optimizer (2 learning rates)
optimizer = torch.optim.AdamW(
    [
        {"params": model.layer4.parameters(), "lr": 1e-4},
        {"params": model.fc.parameters(), "lr": 1e-3},
    ],
    weight_decay=1e-4
)

total_steps = config["epochs"] * config["steps_per_epoch"]

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=total_steps
)



num_classes: 81
first 5 classes: ['airplane', 'apple', 'background', 'backpack', 'banana']
DEVICE: cuda
Model device: cuda:0
Batch device: cuda:0


In [11]:
'''batch = next(iter(train_loader))
x, y = batch
print(x.shape, y.dtype, y.min().item(), y.max().item())

batch = next(iter(val_loader))
xv, yv = batch
print(xv.shape, yv.dtype, yv.min().item(), yv.max().item())'''

'batch = next(iter(train_loader))\nx, y = batch\nprint(x.shape, y.dtype, y.min().item(), y.max().item())\n\nbatch = next(iter(val_loader))\nxv, yv = batch\nprint(xv.shape, yv.dtype, yv.min().item(), yv.max().item())'

In [12]:
@torch.no_grad()
def validate(model, loader, criterion, cfg, global_step):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    progress_bar = tqdm(loader, desc="Validation", mininterval=2.0)
    for x, y in progress_bar:
        x, y = x.to(DEVICE), y.to(DEVICE)

        logits = model(x)
        loss = criterion(logits, y)

        running_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += x.size(0)

        progress_bar.set_postfix(
            loss=running_loss / total,
            acc=correct / total
        )

    val_loss = running_loss / max(total, 1)
    val_acc = correct / max(total, 1)

    return val_loss, val_acc


def train(model, loader, criterion, optimizer, scheduler, cfg, epoch, global_step):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    progress_bar = tqdm(
        loader,
        total=cfg["steps_per_epoch"],
        desc=f"Epoch {epoch+1}/{cfg['epochs']}",
        mininterval=2.0
    )

    for step, (x, y) in enumerate(progress_bar):
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)

        if not torch.isfinite(loss):
            print("⚠️ Non-finite loss, skipping step")
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        scheduler.step()  

        running_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += x.size(0)

        progress_bar.set_postfix(
            loss=running_loss / total,
            acc=correct / total
        )

        global_step += 1

        if step + 1 >= cfg["steps_per_epoch"]:
            break

    train_loss = running_loss / max(total, 1)
    train_acc = correct / max(total, 1)
    return train_loss, train_acc, global_step


In [ ]:
def save_checkpoint(model, optimizer, epoch, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
        },
        path
    )


global_step = 0
best_val_acc = 0.0

print("Starting training...")
print("CUDA available:", torch.cuda.is_available())
print("Model device:", next(model.parameters()).device)

for epoch in range(config["epochs"]):


    train_loss, train_acc, global_step = train(
        model,
        train_loader,
        criterion,
        optimizer,
        scheduler,   
        config,
        epoch,
        global_step,
    )

    # --- Validate
    val_loss, val_acc = validate(
        model,
        val_loader,
        criterion,
        config,
        global_step,
    )

    print(
        f"\nEpoch {epoch+1}/{config['epochs']} Summary:\n"
        f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.3f}\n"
        f"  Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.3f}\n"
    )

    # --- Save best model only
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        print(f"✅ New best validation accuracy: {best_val_acc:.4f}")
        save_checkpoint(
            model,
            optimizer,
            epoch,
            "checkpoints/best_model.pth"
        )

print("Training completed!")

# --- Final save
final_path = "/Users/cyprienvial/Documents/3A/Image/Etude technique /checkpoints/resnet50_coco_background_final.pth"
torch.save(model.state_dict(), final_path)



Starting training...
CUDA available: True
Model device: cuda:0


Epoch 1/2:  90%|█████████ | 9/10 [05:44<00:38, 38.27s/it, acc=0.163, loss=3.6]   
Validation: 5it [09:15, 111.01s/it, acc=0.225, loss=4.18]



Epoch 1/2 Summary:
  Train Loss: 3.5981 | Train Acc: 0.163
  Val   Loss: 4.1850 | Val   Acc: 0.225

✅ New best validation accuracy: 0.2250


Epoch 2/2:  90%|█████████ | 9/10 [05:01<00:33, 33.54s/it, acc=0.263, loss=2.64] 
Validation: 0it [00:00, ?it/s]